# TNMRControl Dashboard Parameters

Open a TNMR `.tnr` or `.tnt` file, ask TNMR for the active Dashboard parameter tabs, extract each tab's parameters, modify selected values, and optionally save.

## 1. Setup

In [ ]:
from pathlib import Path
import ctypes
import importlib
import sys

redstone_dir = next(p / "hardware_capture" / "rf" / "redstone" for p in (Path.cwd(), *Path.cwd().parents) if (p / "hardware_capture" / "rf" / "redstone").is_dir())
if str(redstone_dir) not in sys.path:
    sys.path.insert(0, str(redstone_dir))

import tnmrcontrol
importlib.reload(tnmrcontrol)

from tnmrcontrol import TNMRControl

print("Python running as admin:", bool(ctypes.windll.shell32.IsUserAnAdmin()))

## 2. Select TNMR File

In [ ]:
# Use a .tnr file if TNMR opens it in your installation.
# input_file = Path(r"C:\TNMR\data\common\your_file.tnr")

input_file = Path(r"C:\TNMR\data\common\20260617_extract_waveform_phase_mod_corrected_FID.tnt")

assert input_file.exists(), input_file
input_file

## 3. Connect and Open

In [ ]:
control = TNMRControl()
control.connect()
control.open(input_file)

print("Opened:", control.path)
print("Application:", control.application_info().get("application_object"))

## 4. Ask TNMR for Dashboard Tabs

In [ ]:
DASHBOARD_TAB_ORDER = tnmrcontrol.DASHBOARD_TAB_ORDER

page_names = control.parameter_page_list()

print("TNMR Dashboard pages:")
for name in page_names:
    print(" -", name)

selected_tabs = [tab for tab in DASHBOARD_TAB_ORDER if tab in page_names]
selected_tabs.extend(tab for tab in page_names if tab not in selected_tabs)

print("\nReading tabs:")
for name in selected_tabs:
    print(" -", name)

## 5. Read TNMR Dashboard Parameter Lists

In [ ]:
dashboard_parameters = {
    tab: control.parameter_list_in_page(tab)
    for tab in selected_tabs
}

for tab, names in dashboard_parameters.items():
    print(f"{tab}: {len(names)} parameters")
    for name in names:
        print("  -", name)

## 6. Extract All Tab Parameters

In [ ]:
def short_error(exc):
    text = str(exc)
    if "typed the parameter name correctly" in text:
        return "unavailable for this document or Dashboard spelling"
    return text


def read_parameter(name):
    try:
        return True, control.get_parameter(name)
    except RuntimeError as exc:
        return False, short_error(exc)


dashboard_values = {}
dashboard_unavailable = {}

for tab, names in dashboard_parameters.items():
    values = {}
    unavailable = {}
    for name in names:
        ok, value = read_parameter(name)
        if ok:
            values[name] = value
        else:
            unavailable[name] = value
    dashboard_values[tab] = values
    dashboard_unavailable[tab] = unavailable

for tab in selected_tabs:
    print(f"\n[{tab}]")
    for name in dashboard_parameters.get(tab, []):
        if name in dashboard_values[tab]:
            print(f"{name}: {dashboard_values[tab][name]}")
        else:
            print(f"{name}: <unavailable>")

## 7. Extraction Summary

In [ ]:
for tab in selected_tabs:
    total = len(dashboard_parameters.get(tab, []))
    ok_count = len(dashboard_values.get(tab, {}))
    missing = list(dashboard_unavailable.get(tab, {}))
    print(f"{tab}: {ok_count}/{total} available")
    if missing:
        print("  unavailable:", ", ".join(missing))

## 8. Sequence Tables

In [ ]:
try:
    print("Current sequence:", control.sequence_name())
except RuntimeError as exc:
    print("Sequence name unavailable:", exc)

print("\nSequence page parameters from TNMR:")
for name in dashboard_parameters.get("Sequence", []):
    value = dashboard_values.get("Sequence", {}).get(name, "<unavailable>")
    print(f"{name}: {value}")

try:
    table_names = control.get_table_list()
except RuntimeError as exc:
    table_names = []
    print("Table list unavailable:", exc)

print("Sequence tables:", len(table_names))
for name in table_names:
    print(" -", name)

# Optional: inspect one table.
# table_name = table_names[0]
# print(table_name, control.get_table(table_name))

## 9. Modify Parameters

In [ ]:
PARAMETER_UPDATES = {
    # "Scans 1D": 4,
    # "Observe Freq.": 400.13,
    # "SW +/-": 100000,
    "P(90)": "1.78u",
}

for name, value in PARAMETER_UPDATES.items():
    before_ok, before = read_parameter(name)
    control.set_parameter(name, value)
    after_ok, after = read_parameter(name)
    print(f"{name}: {before} -> {after}")

print("Parameter updates applied:", len(PARAMETER_UPDATES))

## 10. Optional Save

In [ ]:
# Save in place:
control.save()

# Safer first test: save to a copy.
output_file = input_file.with_name(input_file.stem + "_edited" + input_file.suffix)
control.save_as(output_file)
print("Saved copy:", output_file)